<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №5


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:

using System;
using System.Collections.Generic;

// ================== ИНТЕРФЕЙСЫ (Sprint 3) ==================
public interface IDigitalMedia { string FileFormat { get; set; } void Download(); }
public interface IAudioBook    { int DurationMinutes { get; set; } void Listen(); }
public interface IInteractive  { void LaunchInteractiveMode(); }

// ================== SPRINT 4: НОВЫЕ ИНТЕРФЕЙСЫ ==================
public interface ILoanable
{
    bool IsOnLoan { get; }
    DateTime? DueDate { get; }
    void CheckOut(string readerName, int days);
    void CheckIn();
}

public interface IReservable
{
    int ReservationCount { get; }
    void Reserve(string readerName);
    void CancelReservation(string readerName);
    IReadOnlyList<string> GetReservations();
}

public interface ICertifiable
{
    bool IsCertified { get; }
    string CertificationBody { get; }
    void Certify(string body);
    string GetCertificateInfo();
}

public interface IAdaptable
{
    bool HasAdaptation { get; }
    string AdaptationType { get; }
    void AnnounceAdaptation(string type);
    string GetAdaptationInfo();
}

public interface ICitable
{
    int CitationCount { get; }
    void AddCitation(int count);
    string GetCitationInfo();
}

// Sprint 4: контракт для сервиса уведомлений (Dependency Injection)
public interface INotificationService
{
    void Notify(string message);
    void NotifyError(string message);
}

// ================== Book (Sprint 3 + Sprint 4) ==================
public class Book : ILoanable, IReservable
{
    // --- Sprint 3 поля ---
    private string _title; private string _author;
    private int _yearOfPublication; private string _isbn;
    private int _pages; private string _publisher;
    private bool _isAvailable; private string _language;
    private double _rating; private decimal _price; private bool _isDamaged;

    // --- SPRINT 4: новые атрибуты ---
    private string _shelfLocation;
    private DateTime _addedDate;
    private List<string> _reviews = new List<string>();
    private int _maxBorrowDays;

    // --- SPRINT 4: состояние для явной реализации ILoanable/IReservable ---
    private bool _isOnLoan;
    private DateTime? _dueDate;
    private List<string> _reservations = new List<string>();

    // Sprint 3 свойства
    public string Title { get => _title; set => _title = value; }
    public string Author { get => _author; set => _author = value; }
    public int YearOfPublication { get => _yearOfPublication; set => _yearOfPublication = value; }
    public string ISBN { get => _isbn; set => _isbn = value; }
    public int Pages { get => _pages; set => _pages = value; }
    public string Publisher { get => _publisher; set => _publisher = value; }
    public bool IsAvailable { get => _isAvailable; set => _isAvailable = value; }
    public string Language { get => _language; set => _language = value; }
    public double Rating { get => _rating; set => _rating = value; }
    public decimal Price { get => _price; set => _price = value; }
    public bool IsDamaged { get => _isDamaged; set => _isDamaged = value; }

    // SPRINT 4: новые свойства
    public string ShelfLocation { get => _shelfLocation; set => _shelfLocation = value; }
    public DateTime AddedDate { get => _addedDate; set => _addedDate = value; }
    public int MaxBorrowDays { get => _maxBorrowDays; set => _maxBorrowDays = value; }
    public int ReviewCount => _reviews.Count;

    public Book(string title, string author, int yearOfPublication,
                string isbn = "N/A", int pages = 0,
                string publisher = "Неизвестно", bool isAvailable = true,
                string language = "Русский", double rating = 5.0,
                decimal price = 0, bool isDamaged = false,
                // SPRINT 4:
                string shelfLocation = "A-01", int maxBorrowDays = 14)
    {
        Title = title; Author = author; YearOfPublication = yearOfPublication;
        ISBN = isbn; Pages = pages; Publisher = publisher;
        IsAvailable = isAvailable; Language = language; Rating = rating;
        Price = price; IsDamaged = isDamaged;
        // SPRINT 4:
        ShelfLocation = shelfLocation; MaxBorrowDays = maxBorrowDays; AddedDate = DateTime.Now;
    }

    public virtual string GetFullDescription()
        => $"[Книга] \"{Title}\" | Автор: {Author} | Язык: {Language} | Рейтинг: {Rating}";
    public virtual string GetInfo()
        => $"Название: \"{Title}\", Автор: {Author}, Год: {YearOfPublication}, ISBN: {ISBN}, Цена: {Price} руб.";
    public decimal CalculateFine(int days) => CalculateFine(days, 10.0m);
    public decimal CalculateFine(int days, decimal dailyRate)
        => !IsDamaged ? 0 : days * dailyRate;
    public void ApplyDiscount(decimal pct)
    {
        Price -= Price * (pct / 100m);
        Console.WriteLine($"На книгу \"{Title}\" применена скидка. Новая цена: {Price} руб.");
    }
    public virtual void Read() => Console.WriteLine($"Чтение книги \"{Title}\"...");
    public virtual void Borrow()
    {
        if (!IsAvailable) { Console.WriteLine($"Книга \"{Title}\" недоступна."); return; }
        IsAvailable = false;
        Console.WriteLine($"Книга \"{Title}\" выдана.");
    }

    // ===== SPRINT 4: новые методы =====
    public void AddReview(string review)
    {
        _reviews.Add(review);
        Console.WriteLine($"[Book] Отзыв добавлен к \"{Title}\". Всего: {_reviews.Count}");
    }
    public string GetDetailedStatus()
        => $"\"{Title}\" | Полка: {ShelfLocation} | На руках: {_isOnLoan} | Броней: {_reservations.Count} | Отзывов: {_reviews.Count}";
    public string GetReviewsSummary() => $"Книга \"{Title}\": {_reviews.Count} отзыв(ов).";

    // ===== SPRINT 4: ЯВНАЯ реализация ILoanable =====
    bool ILoanable.IsOnLoan => _isOnLoan;
    DateTime? ILoanable.DueDate => _dueDate;
    void ILoanable.CheckOut(string readerName, int days)
    {
        if (_isOnLoan) { Console.WriteLine($"[ILoanable] \"{Title}\" уже выдана."); return; }
        _isOnLoan = true;
        _dueDate = DateTime.Now.AddDays(days);
        IsAvailable = false;
        Console.WriteLine($"[ILoanable] \"{Title}\" выдана {readerName} до {_dueDate:dd.MM.yyyy}.");
    }
    void ILoanable.CheckIn()
    {
        if (!_isOnLoan) { Console.WriteLine($"[ILoanable] \"{Title}\" уже в библиотеке."); return; }
        _isOnLoan = false; _dueDate = null; IsAvailable = true;
        Console.WriteLine($"[ILoanable] \"{Title}\" возвращена.");
    }

    // ===== SPRINT 4: ЯВНАЯ реализация IReservable =====
    int IReservable.ReservationCount => _reservations.Count;
    void IReservable.Reserve(string readerName)
    {
        if (_reservations.Contains(readerName))
        { Console.WriteLine($"[IReservable] {readerName} уже забронировал \"{Title}\"."); return; }
        _reservations.Add(readerName);
        Console.WriteLine($"[IReservable] {readerName} забронировал \"{Title}\". Всего: {_reservations.Count}");
    }
    void IReservable.CancelReservation(string readerName)
    {
        if (_reservations.Remove(readerName))
            Console.WriteLine($"[IReservable] Бронь {readerName} на \"{Title}\" отменена.");
        else
            Console.WriteLine($"[IReservable] Бронь {readerName} не найдена.");
    }
    IReadOnlyList<string> IReservable.GetReservations() => _reservations.AsReadOnly();
}

// ================== Textbook (Sprint 3 + Sprint 4) ==================
public class Textbook : Book, ICertifiable
{
    private string _subject; private string _curriculumStandard; private int _gradeLevel;

    // SPRINT 4: новые атрибуты
    private bool _isApproved; private int _editionNumber;
    private string _onlineResourceUrl; private string _certificationBody;

    public string Subject { get => _subject; set => _subject = value; }
    public string CurriculumStandard { get => _curriculumStandard; set => _curriculumStandard = value; }
    public int GradeLevel { get => _gradeLevel; set => _gradeLevel = value; }

    // SPRINT 4 свойства
    public bool IsApproved { get => _isApproved; set => _isApproved = value; }
    public int EditionNumber { get => _editionNumber; set => _editionNumber = value; }
    public string OnlineResourceUrl { get => _onlineResourceUrl; set => _onlineResourceUrl = value; }

    public Textbook(string title, string author, int year, string subject, int gradeLevel, string curriculum,
        // SPRINT 4:
        bool isApproved = false, int editionNumber = 1, string onlineResourceUrl = "")
        : base(title, author, year)
    {
        Subject = subject; GradeLevel = gradeLevel; CurriculumStandard = curriculum;
        IsApproved = isApproved; EditionNumber = editionNumber; OnlineResourceUrl = onlineResourceUrl;
    }

    public override string GetFullDescription()
        => $"[Учебник] \"{Title}\" | Предмет: {Subject} | Класс: {GradeLevel} | Стандарт: {CurriculumStandard}";
    public void AssignHomework(int exerciseCount)
        => Console.WriteLine($"По учебнику \"{Title}\" задано {exerciseCount} упражнений.");
    public override void Read()
        => Console.WriteLine($"Чтение учебника \"{Title}\" по предмету: {Subject}.");

    // ===== SPRINT 4: новые методы =====
    public void Approve()
    { IsApproved = true; Console.WriteLine($"[Textbook] \"{Title}\" одобрен."); }
    public string GetOnlineResource()
        => string.IsNullOrEmpty(OnlineResourceUrl)
            ? $"У учебника \"{Title}\" нет онлайн-ресурса."
            : $"Онлайн-ресурс для \"{Title}\": {OnlineResourceUrl}";
    public void GeneratePracticeProblems(int count)
        => Console.WriteLine($"[Textbook] Сгенерировано {count} задач по \"{Subject}\".");

    // ===== SPRINT 4: ЯВНАЯ реализация ICertifiable =====
    bool ICertifiable.IsCertified => _isApproved;
    string ICertifiable.CertificationBody => _certificationBody;
    void ICertifiable.Certify(string body)
    {
        _certificationBody = body; _isApproved = true;
        Console.WriteLine($"[ICertifiable] Учебник \"{Title}\" сертифицирован: {body}");
    }
    string ICertifiable.GetCertificateInfo()
        => _isApproved ? $"Сертификат \"{Title}\": {_certificationBody}."
                       : $"Учебник \"{Title}\" не сертифицирован.";
}

// ================== Fiction (Sprint 3 + Sprint 4) ==================
public class Fiction : Book, IAdaptable
{
    private string _genre; private string _mainCharacter; private bool _hasFilmAdaptation;

    // SPRINT 4: новые атрибуты
    private string _seriesName; private List<string> _awards = new List<string>();
    private string _originalLanguage; private bool _isBestseller;

    public string Genre { get => _genre; set => _genre = value; }
    public string MainCharacter { get => _mainCharacter; set => _mainCharacter = value; }
    public bool HasFilmAdaptation { get => _hasFilmAdaptation; set => _hasFilmAdaptation = value; }

    // SPRINT 4 свойства
    public string SeriesName { get => _seriesName; set => _seriesName = value; }
    public string OriginalLanguage { get => _originalLanguage; set => _originalLanguage = value; }
    public bool IsBestseller { get => _isBestseller; set => _isBestseller = value; }
    public int AwardCount => _awards.Count;

    public Fiction(string title, string author, int year, string genre, string mainCharacter, bool hasFilm,
        // SPRINT 4:
        string seriesName = "", string originalLanguage = "Русский")
        : base(title, author, year)
    {
        Genre = genre; MainCharacter = mainCharacter; HasFilmAdaptation = hasFilm;
        SeriesName = seriesName; OriginalLanguage = originalLanguage;
    }

    public override string GetFullDescription()
        => $"[Худ. литература] \"{Title}\" | Жанр: {Genre} | Герой: {MainCharacter} | Экранизация: {(HasFilmAdaptation ? "Да" : "Нет")}";
    public void ReadChapter(int chapter)
        => Console.WriteLine($"Чтение {chapter}-й главы \"{Title}\".");
    public override void Borrow()
    {
        if (!IsAvailable) { Console.WriteLine($"Худ. книга \"{Title}\" недоступна."); return; }
        IsAvailable = false;
        Console.WriteLine($"Выдача худ. книги \"{Title}\" в жанре: {Genre}.");
    }

    // ===== SPRINT 4: новые методы =====
    public void AddAward(string award)
    {
        _awards.Add(award);
        Console.WriteLine($"[Fiction] \"{Title}\" получила награду: {award}. Всего: {_awards.Count}");
    }
    public string GetSeriesInfo()
        => string.IsNullOrEmpty(SeriesName)
            ? $"\"{Title}\" — самостоятельное произведение."
            : $"\"{Title}\" — часть серии \"{SeriesName}\".";
    public void MarkAsBestseller()
    { _isBestseller = true; Console.WriteLine($"[Fiction] \"{Title}\" — бестселлер!"); }
    public string GetAwardsSummary()
        => _awards.Count == 0 ? $"У \"{Title}\" нет наград."
                              : $"Награды \"{Title}\": {string.Join(", ", _awards)}";

    // ===== SPRINT 4: ЯВНАЯ реализация IAdaptable =====
    bool IAdaptable.HasAdaptation => _hasFilmAdaptation;
    string IAdaptable.AdaptationType => _hasFilmAdaptation ? "Фильм" : "Нет";
    void IAdaptable.AnnounceAdaptation(string type)
    {
        _hasFilmAdaptation = true;
        Console.WriteLine($"[IAdaptable] Объявлена экранизация \"{Title}\": {type}");
    }
    string IAdaptable.GetAdaptationInfo()
        => _hasFilmAdaptation ? $"\"{Title}\" экранизирована ({((IAdaptable)this).AdaptationType})."
                              : $"\"{Title}\" пока не экранизирована.";
}

// ================== ScientificLiterature (Sprint 3 + Sprint 4) ==================
public class ScientificLiterature : Book, ICitable
{
    private string _fieldOfScience; private string _doi; private int _impactFactor;

    // SPRINT 4: новые атрибуты
    private int _citationCount; private bool _isPeerReviewed;
    private string _conferenceName; private int _hIndex;

    public string FieldOfScience { get => _fieldOfScience; set => _fieldOfScience = value; }
    public string DOI { get => _doi; set => _doi = value; }
    public int ImpactFactor { get => _impactFactor; set => _impactFactor = value; }

    // SPRINT 4 свойства
    public int CitationCount { get => _citationCount; set => _citationCount = value; }
    public bool IsPeerReviewed { get => _isPeerReviewed; set => _isPeerReviewed = value; }
    public string ConferenceName { get => _conferenceName; set => _conferenceName = value; }
    public int HIndex { get => _hIndex; set => _hIndex = value; }

    public ScientificLiterature(string title, string author, int year, string fieldOfScience,
        string doi, int impactFactor,
        // SPRINT 4:
        int citationCount = 0, bool isPeerReviewed = false,
        string conferenceName = "", int hIndex = 0)
        : base(title, author, year)
    {
        FieldOfScience = fieldOfScience; DOI = doi; ImpactFactor = impactFactor;
        CitationCount = citationCount; IsPeerReviewed = isPeerReviewed;
        ConferenceName = conferenceName; HIndex = hIndex;
    }

    public override string GetFullDescription()
        => $"[Научная литература] \"{Title}\" | Область: {FieldOfScience} | DOI: {DOI} | Импакт: {ImpactFactor}";

    public void AddCitation(int count)
    {
        CitationCount += count;
        Console.WriteLine($"Статья \"{Title}\" процитирована {count} раз(а). Всего: {CitationCount}");
    }

    // ===== SPRINT 4: новые методы =====
    public void PeerReview()
    { _isPeerReviewed = true; Console.WriteLine($"[ScientificLiterature] \"{Title}\" прошла рецензирование."); }
    public int GetCitationCount() => CitationCount;
    public void UpdateHIndex(int newH)
    { _hIndex = newH; Console.WriteLine($"[ScientificLiterature] H-индекс \"{Title}\": {_hIndex}"); }
    public string GetConferenceInfo()
        => string.IsNullOrEmpty(ConferenceName)
            ? $"Статья \"{Title}\" не привязана к конференции."
            : $"Статья \"{Title}\" на конференции: {ConferenceName}";

    // ===== SPRINT 4: ЯВНАЯ реализация ICitable =====
    int ICitable.CitationCount => _citationCount;
    void ICitable.AddCitation(int count)
    {
        _citationCount += count;
        Console.WriteLine($"[ICitable] +{count} цитирований к \"{Title}\" (всего: {_citationCount}).");
    }
    string ICitable.GetCitationInfo()
        => $"Цитирований \"{Title}\": {_citationCount}. H-индекс: {_hIndex}.";
}

// ================== IllustratedTextbook / MultimediaBook (Sprint 3) ==================
public class IllustratedTextbook : Textbook
{
    private string _illustrator;
    public string Illustrator { get => _illustrator; set => _illustrator = value; }
    public IllustratedTextbook(string title, string author, int year, string subject,
        int gradeLevel, string curriculum, string illustrator)
        : base(title, author, year, subject, gradeLevel, curriculum)
    { Illustrator = illustrator; }
    public override string GetFullDescription()
        => base.GetFullDescription() + $" | Иллюстратор: {Illustrator}";
}

public class MultimediaBook : Book, IDigitalMedia, IAudioBook, IInteractive
{
    private string _fileFormat; private int _durationMinutes;
    public string FileFormat { get => _fileFormat; set => _fileFormat = value; }
    public int DurationMinutes { get => _durationMinutes; set => _durationMinutes = value; }
    public MultimediaBook(string title, string author, int year, string fileFormat, int durationMinutes)
        : base(title, author, year)
    { FileFormat = fileFormat; DurationMinutes = durationMinutes; }
    public void Download() => Console.WriteLine($"Скачивание \"{Title}\" ({FileFormat}).");
    public void Listen()   => Console.WriteLine($"Прослушивание \"{Title}\" ({DurationMinutes} мин).");
    public void LaunchInteractiveMode() => Console.WriteLine($"Интерактив \"{Title}\".");
    public override string GetFullDescription()
        => $"[Мультимедиа] \"{Title}\" | Формат: {FileFormat} | {DurationMinutes} мин.";
}

// ================== LibraryCatalog / Reader (Sprint 3) ==================
public class LibraryCatalog<T> where T : Book
{
    private List<T> _items = new List<T>();
    public void AddItem(T item) { _items.Add(item); Console.WriteLine($"[Каталог] + {item.Title}"); }
    public void ShowAll()
    {
        Console.WriteLine($"\n--- Каталог ({typeof(T).Name}) ---");
        foreach (var item in _items) Console.WriteLine(item.GetFullDescription());
    }
}

public class Reader
{
    public string Name { get; set; }
    public Reader(string name) { Name = name; }
    public void TakeBook(Book book)
    { Console.WriteLine($"\n[Читатель {Name}] берет \"{book.Title}\"."); book.Borrow(); }
    public void TakeBook(List<Book> books)
    {
        Console.WriteLine($"\n[Читатель {Name}] берет {books.Count} книг(и).");
        foreach (var b in books) b.Borrow();
    }
}

// ================== SPRINT 4: Управление зависимостями ==================
public class ConsoleNotificationService : INotificationService
{
    public void Notify(string m) => Console.WriteLine($"[INFO] {m}");
    public void NotifyError(string m) => Console.WriteLine($"[ERROR] {m}");
}
public class InMemoryNotificationService : INotificationService
{
    public List<string> Messages { get; } = new List<string>();
    public void Notify(string m) => Messages.Add(m);
    public void NotifyError(string m) => Messages.Add("ERR: " + m);
}

public class LibraryService
{
    private readonly INotificationService _notifier;
    private readonly List<Book> _books = new List<Book>();

    public LibraryService(INotificationService notifier)
        => _notifier = notifier ?? throw new ArgumentNullException(nameof(notifier));

    public void RegisterBook(Book b)
    { _books.Add(b); _notifier.Notify($"Книга \"{b.Title}\" зарегистрирована."); }

    public void LoanBook(Book b, string reader, int days)
    {
        if (!_books.Contains(b)) { _notifier.NotifyError($"\"{b.Title}\" не зарегистрирована."); return; }
        if (b is ILoanable l)
        { l.CheckOut(reader, days); _notifier.Notify($"Выдача \"{b.Title}\" → {reader}."); }
        else _notifier.NotifyError($"\"{b.Title}\" не поддерживает выдачу.");
    }
    public void ReturnBook(Book b)
    {
        if (b is ILoanable l) { l.CheckIn(); _notifier.Notify($"Возврат \"{b.Title}\"."); }
        else _notifier.NotifyError($"\"{b.Title}\" не поддерживает возврат.");
    }
    public void ReserveBook(Book b, string reader)
    {
        if (b is IReservable r) { r.Reserve(reader); _notifier.Notify($"Бронь \"{b.Title}\" → {reader}."); }
        else _notifier.NotifyError($"\"{b.Title}\" не поддерживает бронь.");
    }
    public void ShowRegisteredBooks()
    {
        Console.WriteLine($"\n[LibraryService] Зарегистрировано: {_books.Count}");
        foreach (var b in _books) Console.WriteLine($"  • {b.Title}");
    }
}

// ================================================================
// ============== ВЫПОЛНЕНИЕ (Sprint 3 + Sprint 4) ================
// ================================================================

Console.WriteLine("=== SPRINT 3: демонстрация базовых механизмов ===");

// 1. Generic-класс
LibraryCatalog<Textbook> textbookCatalog = new LibraryCatalog<Textbook>();
var t1 = new Textbook("Алгебра", "Иванов", 2020, "Математика", 9, "ФГОС");
textbookCatalog.AddItem(t1);
textbookCatalog.ShowAll();

LibraryCatalog<Fiction> fictionCatalog = new LibraryCatalog<Fiction>();
var f1 = new Fiction("Гарри Поттер", "Роулинг", 1997, "Фэнтези", "Гарри", true);
fictionCatalog.AddItem(f1);
fictionCatalog.ShowAll();

// 2. Полиморфизм (Overriding)
Console.WriteLine("\n--- Перекрытие методов (Overriding) ---");
Book myBook = new Fiction("Властелин Колец", "Толкин", 1954, "Фэнтези", "Фродо", true);
Console.WriteLine(myBook.GetFullDescription());

// 3. Перегрузка (Overloading)
Console.WriteLine("\n--- Перегрузка методов (Overloading) ---");
myBook.IsDamaged = true;
decimal fine1 = myBook.CalculateFine(5);
decimal fine2 = myBook.CalculateFine(5, 15.5m);
Console.WriteLine($"Штраф (по умолчанию): {fine1} руб.");
Console.WriteLine($"Штраф (свой тариф): {fine2} руб.");

// 4. Перегрузка TakeBook
Console.WriteLine("\n--- Перегрузка TakeBook ---");
Reader reader = new Reader("Алексей");
reader.TakeBook(myBook);
reader.TakeBook(new List<Book> { t1, f1 });


Console.WriteLine("\n\n=== SPRINT 4: явные реализации интерфейсов + DI ===");

// ---------- 1. ЯВНАЯ реализация ILoanable / IReservable (Book) ----------
Console.WriteLine("\n--- 1. ILoanable / IReservable (Book) ---");
Book b1 = new Book("Чистый код", "Мартин", 2008);
ILoanable loan = b1;
IReservable res = b1;

loan.CheckOut("Иван", 14);
Console.WriteLine($"На руках: {loan.IsOnLoan}, до: {loan.DueDate:dd.MM.yyyy}");

res.Reserve("Мария"); res.Reserve("Пётр"); res.Reserve("Мария"); // дубликат
Console.WriteLine($"Броней: {res.ReservationCount}");
res.CancelReservation("Мария");
Console.WriteLine($"После отмены: {res.ReservationCount}");
loan.CheckIn();

// ---------- 2. ЯВНАЯ реализация ICertifiable (Textbook) ----------
Console.WriteLine("\n--- 2. ICertifiable (Textbook) ---");
var tb = new Textbook("Физика", "Петров", 2022, "Физика", 10, "ФГОС");
ICertifiable cert = tb;
cert.Certify("МинОбр");
Console.WriteLine(cert.GetCertificateInfo());
tb.Approve();
tb.GeneratePracticeProblems(15);

// ---------- 3. ЯВНАЯ реализация IAdaptable (Fiction) ----------
Console.WriteLine("\n--- 3. IAdaptable (Fiction) ---");
var novel = new Fiction("Дюна", "Герберт", 1965, "Фантастика", "Пол", false, "Дюна", "Английский");
IAdaptable ad = novel;
ad.AnnounceAdaptation("Сериал");
Console.WriteLine(ad.GetAdaptationInfo());
novel.AddAward("Хьюго"); novel.AddAward("Небьюла");
novel.MarkAsBestseller();
Console.WriteLine(novel.GetAwardsSummary());
Console.WriteLine(novel.GetSeriesInfo());

// ---------- 4. ЯВНАЯ реализация ICitable (ScientificLiterature) ----------
Console.WriteLine("\n--- 4. ICitable (ScientificLiterature) ---");
var sci = new ScientificLiterature("Квантовая физика", "Смирнов", 2021, "Физика", "10.1000/xyz", 5);
sci.PeerReview(); sci.UpdateHIndex(12);
ICitable cite = sci;
cite.AddCitation(10); cite.AddCitation(5);
Console.WriteLine(cite.GetCitationInfo());

// ---------- 5. Dependency Injection ----------
Console.WriteLine("\n--- 5. Dependency Injection (INotificationService) ---");
INotificationService consoleNotifier = new ConsoleNotificationService();
var service = new LibraryService(consoleNotifier);

var sci2 = new ScientificLiterature("Квантовая механика", "Иванов", 2023, "Физика", "10.1000/abc", 6);
service.RegisterBook(sci2);
service.LoanBook(sci2, "Ольга", 21);
service.ReserveBook(sci2, "Сергей");
service.ReturnBook(sci2);
service.ShowRegisteredBooks();

// ---------- 6. Подмена зависимости (для тестов) ----------
Console.WriteLine("\n--- 6. Подмена зависимости (InMemory notifier) ---");
var memNotifier = new InMemoryNotificationService();
var service2 = new LibraryService(memNotifier);
var f2 = new Fiction("Метро 2033", "Глуховский", 2005, "Постапокалипсис", "Артём", false);
service2.RegisterBook(f2);
service2.LoanBook(f2, "Дмитрий", 7);
Console.WriteLine($"Сообщений через in-memory notifier: {memNotifier.Messages.Count}");
foreach (var m in memNotifier.Messages) Console.WriteLine($"  → {m}");

// ---------- 7. Новые атрибуты/методы базового класса ----------
Console.WriteLine("\n--- 7. Новые атрибуты Book ---");
sci.AddReview("Отличная статья!");
sci.AddReview("Полезно для диплома.");
Console.WriteLine(sci.GetDetailedStatus());
Console.WriteLine(sci.GetReviewsSummary());
Console.WriteLine(sci.GetConferenceInfo());

=== SPRINT 3: демонстрация базовых механизмов ===
[Каталог] + Алгебра

--- Каталог (Textbook) ---
[Учебник] "Алгебра" | Предмет: Математика | Класс: 9 | Стандарт: ФГОС
[Каталог] + Гарри Поттер

--- Каталог (Fiction) ---
[Худ. литература] "Гарри Поттер" | Жанр: Фэнтези | Герой: Гарри | Экранизация: Да

--- Перекрытие методов (Overriding) ---
[Худ. литература] "Властелин Колец" | Жанр: Фэнтези | Герой: Фродо | Экранизация: Да

--- Перегрузка методов (Overloading) ---
Штраф (по умолчанию): 50,0 руб.
Штраф (свой тариф): 77,5 руб.

--- Перегрузка TakeBook ---

[Читатель Алексей] берет "Властелин Колец".
Выдача худ. книги "Властелин Колец" в жанре: Фэнтези.

[Читатель Алексей] берет 2 книг(и).
Книга "Алгебра" выдана.
Выдача худ. книги "Гарри Поттер" в жанре: Фэнтези.


=== SPRINT 4: явные реализации интерфейсов + DI ===

--- 1. ILoanable / IReservable (Book) ---
[ILoanable] "Чистый код" выдана Иван до 18.11.2026.
На руках: True, до: 18.11.2026
[IReservable] Мария забронировал "Чистый код". В